<a href="https://colab.research.google.com/github/nitin04-stack/CNN-For-CIFAR/blob/main/CNN_FOR_CIFAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets import CIFAR10

In [ ]:
from torch.utils.data import DataLoader
from torchvision.transforms import transforms


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
traindataset = CIFAR10(root = "/content/drive/MyDrive/Colab Notebooks./data",train = True,download = True,transform = transform)
testdataset = CIFAR10(root = "/content/drive/MyDrive/Colab Notebooks./data",train = False,download = True,transform = transform)

In [ ]:
traindataset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: /content/drive/MyDrive/Colab Notebooks./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [ ]:
testdataset

Dataset CIFAR10
    Number of datapoints: 10000
    Root location: /content/drive/MyDrive/Colab Notebooks./data
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [ ]:
trainloader = DataLoader(traindataset,batch_size = 64,shuffle = True)
testloader = DataLoader(testdataset,batch_size = 64)

In [ ]:
class CNN(nn.Module):
  def __init__(self):
    super(CNN,self).__init__()
    self.conv_layers = nn.Sequential(
        nn.Conv2d(3,32,kernel_size = 3,padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(32,64,kernel_size = 3,padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64,128,kernel_size = 3,padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
    )

    self.fc_layers = nn.Sequential(
        nn.Linear(4*4*128,256),
        nn.ReLU(),

        nn.Linear(256,10)
    )
  def forward(self,x):
    x = self.conv_layers(x)
    x = x.view(x.size(0),-1)  #to make x falatten
    x = self.fc_layers(x)

    return x

In [ ]:
model = CNN()

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
# traning the Cnn

In [ ]:
epochs = 10
train_loss = []
val_loss = []
best_val_loss = float("inf")

for epoch in range(epochs):
  model.train()
  training_loss = 0.0
  running_val_loss = 0.0

  for images , labels in trainloader:
    optimizer.zero_grad()
    output = model.forward(images)
    loss = criterion(output,labels)
    loss.backward()
    optimizer.step()
    training_loss += loss.item()
    epoch_train_loss = training_loss/len(trainloader)
    train_loss.append(epoch_train_loss)

  model.eval()
  with torch.no_grad():
    for images , labels in testloader:
      output = model.forward(images)
      loss = criterion(output,labels)
      running_val_loss += loss.item()
      epoch_val_loss = training_loss/len(testloader)
      val_loss.append(epoch_val_loss)

  print(f"epochs {epoch+1}/{epochs} =>> training_loss is {epoch_train_loss} , validation loss {epoch_val_loss}")
  if epoch_val_loss < best_val_loss:
    best_val_loss = epoch_val_loss
    torch.save(model.state_dict(),"/content/drive/MyDrive/Colab Notebooks/best_model.pt")


epochs 1/10 =>> training_loss is 1.3817265292872554 , validation loss 6.882230228679195
epochs 2/10 =>> training_loss is 0.9475843950610636 , validation loss 4.719815267119438
epochs 3/10 =>> training_loss is 0.7566845383485565 , validation loss 3.7689637515195615
epochs 4/10 =>> training_loss is 0.6305657482284415 , validation loss 3.140779714105995
epochs 5/10 =>> training_loss is 0.5236057603107694 , validation loss 2.608023595942813
epochs 6/10 =>> training_loss is 0.4316434359268459 , validation loss 2.1499692158904047
epochs 7/10 =>> training_loss is 0.34885592115542774 , validation loss 1.7376135690671624
epochs 8/10 =>> training_loss is 0.27221291124477714 , validation loss 1.3558630356268517
epochs 9/10 =>> training_loss is 0.21032904316683101 , validation loss 1.0476261895316041
epochs 10/10 =>> training_loss is 0.1654336603877642 , validation loss 0.824007149192558


In [ ]:
correct_labels = 0
total_labels = 0

with torch.no_grad():
  for images,labels in testloader:
    output = model.forward(images)
    _,predicted = torch.max(output,1)

    correct_labels += (predicted == labels).sum().item()
    total_labels += labels.size(0)
print(f"accuracy_score {correct_labels / total_labels *100}")

accuracy_score 76.03
